[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/agent-observability-course/13_costes_finops/13_proyecto_costes_finops.ipynb)

# Proyecto 13 · Panel de FinOps con coste por ticket, presupuesto y alerta de desvío

| | |
|---|---|
| **Nivel** | 🟠 Avanzado |
| **Duración estimada** | ≈ 1 h 45 min |
| **Servicios de Compose** | Ninguno (todo se evalúa en memoria). Opcional: `core` + `metrics` para ver tu panel funcionando (paso final) |
| **RAM aproximada** | < 500 MB |
| **Prerrequisitos** | Lección `13_costes_finops`, módulo 10 (dashboards como código con `supportops.dashkit`) y proyecto 11 (cascadas y costes por agente) |
| **Conexión** | Convierte la telemetría de SupportOps en **control financiero**: cuánto cuesta cada ticket, quién lo genera, cuándo nos pasamos y cómo nos enteramos antes de que llegue la factura |

> ⚠️ **Todos los precios de este módulo son ILUSTRATIVOS** (no son de ningún proveedor ni de ninguna fecha): sirven para que las magnitudes sean razonables. Cárgalos de tu proveedor con `PriceTable.from_json(...)` o `SUPPORTOPS_PRICES_FILE`.

## Contexto de negocio

Eres ingeniero/a de plataforma en **SupportOps**. Dirección pide tres cosas antes de aprobar el presupuesto de IA del trimestre:

1. **Un número fiable de coste por ticket resuelto**, con quién lo genera (cliente, tier, *feature*).
2. **Un tope** que evite que un solo ticket (un bucle) gaste una barbaridad, sin bloquear a los tickets normales.
3. **Una alerta de desvío** que salte en minutos —no a fin de mes— si el coste por ticket se dispara (alguien cambió el modelo, se rompió la caché…), y un **panel** que lo muestre.

## Datos

Todo sintético y determinista (semilla 42, `mock`, tiempo simulado, **precios ilustrativos**): el **Paso 0** ejecuta 240 tickets con un 6 % de bucles de herramientas y un prompt de sistema largo (≈ 2 500 tokens), y construye el **libro de costes** (`LEDGER`: una fila por ticket) con el `FinOpsInstrumentor` de la lección. También genera una **serie de coste por ticket** por minuto de un incidente de 90 min (a partir del minuto 60 alguien cambia el modelo por defecto). Cadena de datos: stack local (opcional) → memoria.

## Entregables y criterios de evaluación (rúbrica)

| # | Entregable | Criterio medido | Umbral (calibrado con la solución; `mock`, semilla 42) |
|---|---|---|---|
| R1 | `cost_breakdown` | 7 vectores: semántica de subconjuntos de caché, estilo Anthropic, sin multiplicadores, invariantes y modelo desconocido | 7/7 |
| R2 | `attribution` | % del gasto del 5 % de clientes con más coste, coste por ticket por tier, coste por ticket **resuelto**, gasto desperdiciado y reparto por *feature* | exacto (tol. 1e-9) |
| R3 | `PerTicketCap` | Con `CAP_USD`: 0 falsos bloqueos, bloquea todos los bucles que lo superan, coste máximo ≤ tope + 1 llamada | 3/3 |
| R4 | `loop_report` | Detecta los bucles inyectados con precisión y exhaustividad ≥ 0,95 y calcula el coste desperdiciado | ≥ 0,95 / ± 2 % |
| R5 | `drift_alert` | Sin falsa alarma en la línea base, detecta el incidente en ≤ 4 min, ignora picos de 2 puntos y rachas rotas por huecos | 4/4 |
| R6 | `build_finops_dashboard` | `lint` limpio y las 8 consultas clave (coste por ticket p95, coste por modelo, caché, desperdicio, burn rate, desvío, Loki por cliente, Tempo de tickets caros) | 8/8 |

> 📏 Los umbrales salen de la solución de referencia y se comprobaron con las semillas 42, 7 y 99 (las cifras exactas del R2 son las de la semilla 42, fija en el notebook). El tiempo es simulado: nada depende de tu máquina.

In [ ]:
# Instala el paquete del curso (editable) con extras de gráficos. Sirve en local y en Colab (clona el repo si hace falta).
import pathlib, subprocess, sys

def _find_root():
    for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (p / "pyproject.toml").exists() and (p / "supportops").is_dir():
            return p
    return None

ROOT = _find_root()
if ROOT is None:  # p. ej. Google Colab: el notebook se abre sin el repo
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/padillacm/IA-practice", "/content/IA-practice"], check=True)
    ROOT = pathlib.Path("/content/IA-practice/agent-observability-course")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[viz,otel]"], check=True)
print("Curso en:", ROOT)

In [ ]:
import os
os.environ.setdefault("LLM_PROVIDER", "mock")          # mock = determinista, sin coste, sin API keys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from supportops import *                                 # SupportAgent, generate_tickets, check_stack, ...
from supportops import viz, sample_data, schema, backends
viz.apply_style()
pd.set_option("display.width", 140); pd.set_option("display.max_colwidth", 80)
SEED = 42
FAST_DEV_RUN = True   # True: pocos tickets (iteración rápida). False: tamaños «completos» del texto

import dataclasses, json, math, time
from supportops.instrument import CompositeInstrumentor
from supportops.telemetry import OTelInstrumentor, setup_telemetry
from supportops.finops import (ILLUSTRATIVE_PRICES, BudgetGuard, BudgetPool, CacheSimulator, CostLedger, DowngradeLLM, FinOpsInstrumentor, LoopGuard,
                               ModelPrice, PriceTable, Usage4, cost_drift_alert, detect_loop_in_result, finops_alert_rule_json, policy_prompt, semconv_usage)
STACK = check_stack([])
LIVE = False                      # True: Collector + Prometheus responden y podemos enviar telemetría de verdad
PP = policy_prompt(2500)                       # prompt de sistema largo (políticas incrustadas): ≈ 2 500 tokens, para que la caché importe

from supportops.dashkit import Dashboard, lint, prom, loki, tempo, timeseries, stat, bargauge, traces_table, logs, thresholds
from supportops.pricing import cost_usd

## Paso 0 · Los datos (te lo damos hecho)

El sistema de partida ejecuta 240 tickets con el agente de SupportOps **sin guardas** (para medir el coste «libre», incluidos los bucles) y a la vez construye el libro de costes con caché (`CacheSimulator`) y los precios ilustrativos.

In [ ]:
TK = generate_tickets(240, seed=SEED, fault_rates={**DEFAULT_FAULT_RATES, "tool_loop": 0.06})
_led = CostLedger()
_fin = FinOpsInstrumentor(ILLUSTRATIVE_PRICES, cache=CacheSimulator(min_prefix_tokens=256), ledger=_led, loop_fn=lambda r: detect_loop_in_result(r)[0])
RES_FREE = run_tickets(SupportAgent(system_prompt=PP, instrumentor=_fin), TK)           # agente libre: cada AgentResult.cost_usd usa los precios del mock (sin caché)
LEDGER = _led.to_frame()
print(f"{len(LEDGER)} tickets · coste con caché ${LEDGER.cost_usd.sum():.4f} · sin caché ${sum(r.cost_usd for r in RES_FREE):.4f} · bucles inyectados: {sum('tool_loop' in r.injected_faults for r in RES_FREE)}")
display(LEDGER[["ticket_id", "customer_id", "tier", "feature", "model", "status", "input_total", "cache_read", "output", "cost_usd", "wasted_usd"]].head(4).round(6))

# Serie de un incidente de 90 min: coste por ticket y minuto (NaN en los minutos sin tráfico). A partir del minuto 60 alguien pone el modelo grande por defecto.
def serie_incidente(seed=SEED, minutos=90, onset=60, por_min=3):
    L = CostLedger(); cache = CacheSimulator(min_prefix_tokens=256)
    for m in range(minutos):
        tk = generate_tickets(por_min, seed=seed * 1000 + m, duration_s=60, fault_rates=DEFAULT_FAULT_RATES, start_s=SIM_EPOCH_S + m * 60)
        run_tickets(SupportAgent(llm=MockLLM("mock-large" if m >= onset else "mock-small"), system_prompt=PP, instrumentor=FinOpsInstrumentor(ILLUSTRATIVE_PRICES, cache=cache, ledger=L)), tk)
    d = L.to_frame(); d["minuto"] = ((d.t_start - SIM_EPOCH_S) // 60).astype(int)
    g = d.groupby("minuto").cost_usd.mean().reindex(range(minutos))
    return [float(x) for x in g.values]
SERIE = serie_incidente()
print(f"serie: {len(SERIE)} minutos · mediana de los 60 primeros ${np.nanmedian(SERIE[:60]):.6f} · mediana del incidente ${np.nanmedian(SERIE[60:]):.6f} · ratio {np.nanmedian(SERIE[60:]) / np.nanmedian(SERIE[:60]):.1f}×")

## Paso 1 · De tokens a dinero

**TODO 1.** `cost_breakdown(prices, model, usage, style="semconv")` devuelve un `dict` `{"input", "cache_read", "cache_write", "output", "total"}` en USD.

- `prices`: `dict` modelo → `{"input", "output", "cache_read"?, "cache_write"?}` en USD por **millón** de tokens. Si faltan `cache_read`/`cache_write`, se cobran como `input` (sin descuento ni recargo).
- `usage`: `dict` con `input_tokens`, `output_tokens` y, opcionalmente, `cache_read_input_tokens` y `cache_write_input_tokens` (estilo spec) **o** `cache_creation_input_tokens` (estilo Anthropic).
- `style="semconv"`: `input_tokens` **incluye** la caché. `style="anthropic"`: `input_tokens` **excluye** la caché (el total es `input + lectura + creación`).
- Lanza `ValueError` si `cache_read + cache_write > input_total` (invariante de subconjuntos) o si algún token es negativo, y `KeyError` si el modelo no está en `prices`.

<details>
<summary>🪜 Pista 1</summary>

Primero normaliza a `(input_total, cache_read, cache_write, output)`: para `anthropic`, `input_total = input_tokens + cache_read_input_tokens + cache_creation_input_tokens`. Después valida el invariante y calcula `u = input_total - r - w`.

</details>

<details>
<summary>🪜 Pista 2</summary>

`p = prices[model]` (da `KeyError` solo); `p_cr = p.get('cache_read', p['input'])`, `p_cw = p.get('cache_write', p['input'])`; `input = u * p['input'] / 1e6`, etc., y `total` = suma de los cuatro.

</details>

In [ ]:
PRICES = {"m": {"input": 3.0, "output": 15.0, "cache_read": 0.3, "cache_write": 3.75}, "n": {"input": 1.0, "output": 2.0}}

def cost_breakdown(prices: dict, model: str, usage: dict, style: str = "semconv") -> dict:
    # TODO 1
    raise NotImplementedError("TODO 1")

try:
    print(cost_breakdown(PRICES, "m", {"input_tokens": 1000, "output_tokens": 100, "cache_read_input_tokens": 600, "cache_write_input_tokens": 100}))
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 2 · Atribución y unidad económica

**TODO 2.** `attribution(ledger)` (un `DataFrame` con las columnas `customer_id`, `tier`, `feature`, `status`, `cost_usd`, `wasted_usd`) devuelve un `dict`:

- `top5_share`: fracción del gasto total que generan el **5 % de los clientes con más coste** (nº de clientes = `max(1, round(0.05 · nº de clientes distintos))`).
- `cost_per_ticket_by_tier`: `dict` tier → coste medio por ticket.
- `cost_per_resolved`: gasto **total** dividido por el nº de tickets **resueltos** (`status` ∈ {`ok`, `escalated`}).
- `wasted_share`: `wasted_usd.sum() / cost_usd.sum()`.
- `feature_share`: `dict` *feature* → fracción del gasto total.

<details>
<summary>🪜 Pista 1</summary>

`by_c = ledger.groupby('customer_id').cost_usd.sum().sort_values(ascending=False)`; `k = max(1, round(0.05 * len(by_c)))`; `top5_share = by_c.head(k).sum() / by_c.sum()`.

</details>

In [ ]:
def attribution(ledger: pd.DataFrame) -> dict:
    # TODO 2
    raise NotImplementedError("TODO 2")

try:
    A = attribution(LEDGER); print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in A.items()})
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 3 · Un tope por ticket que no bloquee a los normales

**TODO 3.** Completa `PerTicketCap(cap_usd)`, un `Instrumentor` que **rechaza la siguiente llamada al LLM** cuando el gasto acumulado del ticket ya alcanzó `cap_usd`, lanzando `LLMError("presupuesto agotado", status=429, error_type="budget_exceeded", retryable=False, latency_s=0.0)` **antes** de la llamada. El gasto de cada llamada es el de **tu** `cost_breakdown` con `PRICES_MOCK` (sin caché; usa `style="semconv"`). Guarda el gasto acumulado en `ctx.state["cap_spent"]` y apunta cada rechazo en `self.blocked` (ids de ticket).

El tope `CAP_USD` está calibrado para los tickets normales (≈ p99,9 de su coste **sin caché**): la rúbrica comprueba que **no bloquea ninguno** y que **sí** bloquea los bucles que lo superan.

<details>
<summary>🪜 Pista 1</summary>

`llm_call` es un *context manager*: `@contextmanager def llm_call(self, ctx, info):` con la comprobación **antes** del `yield` (lanza el error) y la acumulación **después** (usa `info.response.usage.input_tokens/output_tokens` y `info.response.model`).

</details>

In [ ]:
from contextlib import contextmanager
from supportops.llm import LLMError
PRICES_MOCK = {"mock-small": {"input": 0.25, "output": 1.25}, "mock-large": {"input": 3.0, "output": 15.0}}
CAP_USD = 0.0030

class PerTicketCap(Instrumentor):
    def __init__(self, cap_usd: float):
        self.cap_usd, self.blocked = cap_usd, []

    @contextmanager
    def llm_call(self, ctx, info):
        # TODO 3
        raise NotImplementedError("TODO 3")
        yield

try:
    _cap = PerTicketCap(CAP_USD)
    _r = run_tickets(SupportAgent(system_prompt=PP, instrumentor=_cap), TK[:60])
    print("bloqueados en los 60 primeros:", len(_cap.blocked))
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 4 · Cuantificar el coste de los bucles

**TODO 4.** `loop_report(results, truth=None)` recibe una lista de `AgentResult` y devuelve `{"flagged": [ticket_id…], "precision", "recall", "wasted_usd"}`:

- Un ticket es un **bucle** si contiene **3 o más llamadas idénticas** a herramientas (mismo nombre y mismos argumentos, en `step.attributes["arguments"]` de los `StepRecord` con `kind == "tool"`).
- `precision`/`recall` frente a la verdad de terreno: `"tool_loop" in r.injected_faults`.
- `wasted_usd`: suma, sobre los tickets marcados, del coste (`supportops.pricing.cost_usd(modelo, entrada, salida)`, precios del mock) de **todos los pasos `llm` posteriores a la 2.ª llamada idéntica** (lo que un `LoopGuard(2)` habría ahorrado). Un paso `llm` es posterior si su `index` es mayor que el de la 2.ª herramienta repetida.

<details>
<summary>🪜 Pista 1</summary>

Para cada resultado cuenta repeticiones con un `dict` clave `(nombre, json.dumps(argumentos, sort_keys=True))`; marca como bucle si el máximo ≥ 3. Para el desperdicio localiza el `index` del paso de herramienta donde esa clave llega a 2.

</details>

In [ ]:
def loop_report(results: list, truth=None) -> dict:
    # TODO 4
    raise NotImplementedError("TODO 4")

try:
    LR = loop_report(RES_FREE); print({k: (round(v, 4) if isinstance(v, float) else (v[:3] if isinstance(v, list) else v)) for k, v in LR.items()})
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 5 · La alerta de desvío

**TODO 5.** `drift_alert(series, baseline_n=30, ratio=1.6, persist=3)` devuelve el **índice** del primer punto en que salta la alerta, o `None`. Regla: la **línea base** es la mediana de los primeros `baseline_n` puntos (ignora `NaN`); salta cuando hay `persist` puntos **consecutivos** con valor ≥ `ratio` × línea base. Un `NaN` (minuto sin tráfico) **rompe** la racha. Empieza a evaluar en el índice `baseline_n`. Si la línea base tiene menos de `max(3, baseline_n // 2)` puntos válidos (no `NaN`), devuelve `None`.

<details>
<summary>🪜 Pista 1</summary>

`import statistics`; `base = [x for x in series[:baseline_n] if x == x]` (el `NaN` no es igual a sí mismo); `umbral = ratio * statistics.median(base)`; recorre con un contador `racha` que se reinicia a 0 con un valor menor o con `NaN`.

</details>

In [ ]:
def drift_alert(series, baseline_n: int = 30, ratio: float = 1.6, persist: int = 3):
    # TODO 5
    raise NotImplementedError("TODO 5")

try:
    print("alerta en el minuto:", drift_alert(SERIE))
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 6 · El panel de FinOps como código

**TODO 6.** `build_finops_dashboard()` devuelve el `dict` de un dashboard (`Dashboard(...).to_dict()` de `supportops.dashkit`, módulo 10) con **uid** `finops`, título «13 · FinOps (proyecto)» y al menos **8 paneles**, cada uno con `description`. Debe pasar `lint(...)` y contener estas **consultas** (las métricas las publica `FinOpsInstrumentor`: ver la lección, sección 8-9):

1. **Coste por ticket, p95**: `histogram_quantile(0.95, …supportops_ticket_cost_USD_bucket…)`.
2. **Coste por modelo**: `sum by (gen_ai_request_model) (rate(supportops_llm_cost_USD_total[…]))`.
3. **Hit ratio de la caché**: una expresión con `supportops_llm_tokens_total` y `cache_read`.
4. **Gasto desperdiciado**: una expresión con `supportops_ticket_cost_wasted_USD_total`.
5. **Burn rate del presupuesto**: `supportops_budget_burn_rate`.
6. **Desvío**: coste por ticket actual / el de **hace 1 h** (`offset 1h`).
7. **Top clientes** (Loki): una consulta que contenga `ticket costed` y `supportops_finops_cost_usd`.
8. **Tickets caros** (Tempo, TraceQL): una consulta que contenga `span.supportops.finops.cost_usd`.

<details>
<summary>🪜 Pista 1</summary>

Estructura (módulo 10): `d = Dashboard('finops', '13 · FinOps (proyecto)', tags=['curso', 'finops'])`; `d.add(stat('…', 'promql', description='…'), timeseries('…', [prom('promql', '{{leyenda}}')], description='…'))`; `d.add(traces_table('…', '{ … }', description='…'))`; `logs(...)` o `table([loki('…')], …)` para Loki; `return d.to_dict()`.

</details>

In [ ]:
def build_finops_dashboard() -> dict:
    # TODO 6
    raise NotImplementedError("TODO 6")

try:
    D = build_finops_dashboard(); print(len(D["panels"]), "paneles · lint:", lint(D) or "OK")
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 7 · Rúbrica

In [ ]:
REF = {"top5_share": 0.43434498924459763, "cost_per_resolved": 0.00031152420454545456, "wasted_share": 0.1657263972994948,
       "by_tier": {"premium": 0.0003021250954198473, "standard": 0.0002656599770642202},
       "feature_share": {"escalation": 0.14457398429204207, "kb_question": 0.3411003741501189, "order_status": 0.3418857355677528, "refund": 0.1724399059900861, "unknown": 0.0},
       "loop_wasted": 0.038409, "n_loops": 12}      # calibrado con la solución (semilla 42)

def run_rubric():
    rows = []
    def add(r, d, ok, det=""): rows.append({"criterio": r, "descripción": d, "resultado": "✅ OK" if ok else "❌ FALLA", "detalle": det})
    ap = lambda a, b, t=1e-12: abs(a - b) <= t * max(1.0, abs(b))
    # R1
    try:
        U = {"input_tokens": 1000, "output_tokens": 100, "cache_read_input_tokens": 600, "cache_write_input_tokens": 100}
        c1 = cost_breakdown(PRICES, "m", U)
        c2 = cost_breakdown(PRICES, "m", {"input_tokens": 300, "output_tokens": 100, "cache_read_input_tokens": 600, "cache_creation_input_tokens": 100}, style="anthropic")
        c3 = cost_breakdown(PRICES, "n", {"input_tokens": 1_000_000, "output_tokens": 0, "cache_read_input_tokens": 500_000})
        def raises(exc, fn):
            try: fn()
            except exc: return True
            except Exception: return False
            return False
        checks = {"subconjuntos": ap(c1["input"], 300 * 3 / 1e6) and ap(c1["cache_read"], 600 * 0.3 / 1e6) and ap(c1["cache_write"], 100 * 3.75 / 1e6) and ap(c1["output"], 100 * 15 / 1e6) and ap(c1["total"], sum(c1[k] for k in ("input", "cache_read", "cache_write", "output"))),
                  "estilo Anthropic = spec": all(ap(c2[k], c1[k]) for k in c1),
                  "sin multiplicadores": ap(c3["total"], 1.0) and ap(c3["cache_read"], 0.5),
                  "invariante": raises(ValueError, lambda: cost_breakdown(PRICES, "m", {"input_tokens": 100, "output_tokens": 1, "cache_read_input_tokens": 80, "cache_write_input_tokens": 40})),
                  "negativos": raises(ValueError, lambda: cost_breakdown(PRICES, "m", {"input_tokens": -1, "output_tokens": 1})),
                  "modelo desconocido": raises(KeyError, lambda: cost_breakdown(PRICES, "zzz", U)),
                  "anthropic invariante": ap(cost_breakdown(PRICES, "n", {"input_tokens": 10, "output_tokens": 0, "cache_read_input_tokens": 90}, style="anthropic")["total"], 100 * 1.0 / 1e6)}
        add("R1", "cost_breakdown: 7 vectores", all(checks.values()), str({k: v for k, v in checks.items() if not v} or "todas OK"))
    except Exception as e: add("R1", "cost_breakdown", False, f"{type(e).__name__}: {e}")
    # R2
    try:
        a = attribution(LEDGER)
        ok = ap(a["top5_share"], REF["top5_share"], 1e-9) and ap(a["cost_per_resolved"], REF["cost_per_resolved"], 1e-9) and ap(a["wasted_share"], REF["wasted_share"], 1e-9) \
            and all(ap(a["cost_per_ticket_by_tier"][k], v, 1e-9) for k, v in REF["by_tier"].items()) and all(ap(a["feature_share"].get(k, -1), v, 1e-9) for k, v in REF["feature_share"].items())
        add("R2", "attribution: top-5 %, tier, resuelto, desperdicio y features", ok, f"top5={a['top5_share']:.4f} (ref {REF['top5_share']:.4f}) · USD/resuelto={a['cost_per_resolved']:.6f} (ref {REF['cost_per_resolved']:.6f}) · desperdicio={a['wasted_share']:.4f} (ref {REF['wasted_share']:.4f})")
    except Exception as e: add("R2", "attribution", False, f"{type(e).__name__}: {e}")
    # R3
    try:
        cap = PerTicketCap(CAP_USD)
        res_c = run_tickets(SupportAgent(system_prompt=PP, instrumentor=cap), TK)
        free = {r.ticket_id: r for r in RES_FREE}; blocked = set(cap.blocked)
        normal_blocked = [t for t in blocked if "tool_loop" not in free[t].injected_faults]
        expected = {t for t, r in free.items() if "tool_loop" in r.injected_faults and r.cost_usd > CAP_USD}
        max_one_call = max(max(cost_usd(s.name, s.input_tokens, s.output_tokens) for s in r.steps if s.kind == "llm") for r in RES_FREE if any(s.kind == "llm" for s in r.steps))
        checks = {"0 falsos bloqueos": len(normal_blocked) == 0, "bloquea los bucles que superan el tope": blocked == expected and len(expected) > 0,
                  "coste ≤ tope + 1 llamada": max(r.cost_usd for r in res_c) <= CAP_USD + max_one_call + 1e-9,
                  "estado de los bloqueados": all(r.status == "error" and r.error_type == "budget_exceeded" for r in res_c if r.ticket_id in blocked)}
        add("R3", "PerTicketCap: 3 comprobaciones + estado", all(checks.values()), f"{ {k: v for k, v in checks.items() if not v} or 'todas OK'} · bloqueados={len(blocked)} (esperados {len(expected)})")
    except Exception as e: add("R3", "PerTicketCap", False, f"{type(e).__name__}: {e}")
    # R4
    try:
        lr = loop_report(RES_FREE)
        ok = lr["precision"] >= 0.95 and lr["recall"] >= 0.95 and abs(lr["wasted_usd"] - REF["loop_wasted"]) <= 0.02 * REF["loop_wasted"] and len(lr["flagged"]) == REF["n_loops"]
        add("R4", "loop_report: precisión/exhaustividad ≥ 0,95 y desperdicio ± 2 %", ok, f"precisión={lr['precision']:.2f} exhaustividad={lr['recall']:.2f} desperdicio=${lr['wasted_usd']:.5f} (ref ${REF['loop_wasted']:.5f}) · marcados={len(lr['flagged'])} (ref {REF['n_loops']})")
    except Exception as e: add("R4", "loop_report", False, f"{type(e).__name__}: {e}")
    # R5
    try:
        rng = np.random.default_rng(1); base = list(1.0 + 0.1 * rng.standard_normal(60))
        checks = {"sin falsa alarma en la base": drift_alert(base + [1.0] * 30) is None,
                  "detecta el incidente real en ≤ 4 min": (lambda i: i is not None and 60 <= i <= 64)(drift_alert(SERIE)),
                  "picos de 2 puntos no alertan": drift_alert(base + [3.0, 3.0, 1.0, 3.0, 3.0, 1.0] + [1.0] * 10) is None,
                  "los huecos (NaN) rompen la racha y el índice es correcto": drift_alert(base + [3.0, 3.0, float("nan"), 3.0, 3.0, 3.0]) == 60 + 5 and drift_alert(base[:10] + [float("nan")] * 20 + [5.0] * 5, baseline_n=30) is None}
        add("R5", "drift_alert: 4 escenarios", all(checks.values()), str({k: v for k, v in checks.items() if not v} or f"todas OK · alerta en el minuto {drift_alert(SERIE)}"))
    except Exception as e: add("R5", "drift_alert", False, f"{type(e).__name__}: {e}")
    # R6
    try:
        import re as _re
        D = build_finops_dashboard()
        exprs = []
        for p in D["panels"]:
            for t in p.get("targets", []): exprs.append(t.get("expr") or t.get("query") or "")
        has = lambda *pats: any(all(_re.search(p, e) for p in pats) for e in exprs)
        checks = {"lint limpio y uid": not lint(D) and D["uid"] == "finops" and len(D["panels"]) >= 8,
                  "p95 coste por ticket": has(r"histogram_quantile\(0\.95", r"supportops_ticket_cost_USD_bucket"), "coste por modelo": has(r"sum by \(gen_ai_request_model\)", r"supportops_llm_cost_USD_total"),
                  "hit ratio": has(r"supportops_llm_tokens_total", r"cache_read"), "desperdicio": has(r"supportops_ticket_cost_wasted_USD_total"), "burn rate": has(r"supportops_budget_burn_rate"),
                  "desvío (offset 1h)": has(r"offset 1h", r"supportops_ticket_cost_USD"), "Loki por cliente": has(r"ticket costed", r"supportops_finops_cost_usd"),
                  "Tempo tickets caros": has(r"span\.supportops\.finops\.cost_usd")}
        add("R6", "build_finops_dashboard: lint + 8 consultas", all(checks.values()), str({k: v for k, v in checks.items() if not v} or "todas OK"))
    except Exception as e: add("R6", "build_finops_dashboard", False, f"{type(e).__name__}: {e}")
    return pd.DataFrame(rows).set_index("criterio")

In [ ]:
# Ejecuta esta celda cuando hayas terminado los TODO
display(run_rubric())

---
## Paso final (opcional) · Míralo funcionar

Con el stack (`docker compose --profile core --profile metrics up -d`): guarda tu dashboard con `Dashboard.save(...)` en `stack/grafana/dashboards/`, envía telemetría con `FinOpsInstrumentor(..., handle=setup_telemetry("supportops-agent"))` y `generate_tickets(..., start_s="live")`, provoca un cambio de modelo a mitad y comprueba que tu panel de desvío cruza el umbral y que `finops_alert_rule_json(...)` (lección, sección 9.1) dispara.

## Retos extra (nivel experto)

1. **Presupuesto por cliente y día** con multiplicador por tier (`BudgetGuard(per_customer_day_usd=…, tier_multiplier={"premium": 2})`): ¿cuántos clientes se bloquean con un tope del p95 diario y cuánto gasto evitas? ¿Qué pasa con los premium?
2. **Degradación inteligente:** `DowngradeLLM` al 25 % del tope frente a bloqueo: mide coste, tickets resueltos y `grounded` (¿baja la calidad del modelo pequeño?).
3. **Reponderación:** vuelve a calcular el gasto del libro con una segunda tabla de precios y explica por qué guardar **tokens** es mejor que guardar dólares.
4. **Alerta multiventana:** combina una ventana corta (5 min) y una larga (1 h) como el *burn rate* de un SLO para reducir falsos positivos; calibra con la línea base de `SERIE` y mide el retraso de detección.
5. **Multi-agente:** reutiliza `MultiAgentSupportOps` (módulo 11), reprecia `per_agent` con `PriceTable` y atribuye el coste por agente y modelo; ¿qué agente domina?

## Reflexión (producción e hilo conductor)

- Tu `CAP_USD` bloqueó bucles pero un bloqueo es un ticket **sin resolver**: ¿qué ocurre con el cliente? ¿Dónde pondrías la **degradación a humano** y cómo medirías su coste (módulo 11, fallback)?
- `drift_alert` salta en 1-3 min con un cambio ×12 y **no** vería un +30 %: ¿qué métrica añadirías (coste desperdiciado, hit ratio, tokens por ticket) para el desvío lento?
- La unidad económica es el **ticket resuelto**: ¿cómo la combinarías con la calidad del módulo 14 (juez) para no «ahorrar» empeorando las respuestas?
- El módulo 18 trata el **coste del propio stack de observabilidad**: ¿cuánto gasta tu telemetría de FinOps (spans, series, logs por ticket) comparado con lo que ahorra?

---
# ⛔ SPOILER — intenta resolverlo primero

A continuación, la **solución de referencia** completa y ejecutable (redefine las funciones y vuelve a lanzar la rúbrica).

In [ ]:
# ===================== SOLUCIÓN DE REFERENCIA =====================
def cost_breakdown(prices, model, usage, style="semconv"):
    p = prices[model]
    cr, cw = int(usage.get("cache_read_input_tokens", 0)), int(usage.get("cache_write_input_tokens", usage.get("cache_creation_input_tokens", 0)))
    inp, out = int(usage["input_tokens"]), int(usage["output_tokens"])
    total_in = inp + cr + cw if style == "anthropic" else inp
    if min(total_in, out, cr, cw) < 0: raise ValueError("tokens negativos")
    if cr + cw > total_in: raise ValueError("cache_read + cache_write > input_total")
    u = total_in - cr - cw
    d = {"input": u * p["input"] / 1e6, "cache_read": cr * p.get("cache_read", p["input"]) / 1e6, "cache_write": cw * p.get("cache_write", p["input"]) / 1e6, "output": out * p["output"] / 1e6}
    d["total"] = sum(d.values())
    return d

def attribution(ledger):
    by_c = ledger.groupby("customer_id").cost_usd.sum().sort_values(ascending=False)
    k = max(1, round(0.05 * len(by_c)))
    resolved = ledger.status.isin(["ok", "escalated"]).sum()
    return {"top5_share": float(by_c.head(k).sum() / by_c.sum()), "cost_per_ticket_by_tier": ledger.groupby("tier").cost_usd.mean().to_dict(),
            "cost_per_resolved": float(ledger.cost_usd.sum() / resolved), "wasted_share": float(ledger.wasted_usd.sum() / ledger.cost_usd.sum()),
            "feature_share": (ledger.groupby("feature").cost_usd.sum() / ledger.cost_usd.sum()).to_dict()}

class PerTicketCap(Instrumentor):
    def __init__(self, cap_usd):
        self.cap_usd, self.blocked = cap_usd, []

    @contextmanager
    def llm_call(self, ctx, info):
        spent = ctx.state.get("cap_spent", 0.0)
        if spent >= self.cap_usd:
            self.blocked.append(ctx.ticket.ticket_id)
            raise LLMError("presupuesto agotado", status=429, error_type="budget_exceeded", retryable=False, latency_s=0.0)
        yield
        r = info.response
        if r is not None:
            usage = {"input_tokens": r.usage.input_tokens, "output_tokens": r.usage.output_tokens}
            ctx.state["cap_spent"] = spent + cost_breakdown(PRICES_MOCK, r.model, usage)["total"]

def loop_report(results, truth=None):
    flagged, wasted = [], 0.0
    for r in results:
        seen, second_idx = {}, None
        for s in r.steps:
            if s.kind != "tool": continue
            k = (s.name, json.dumps(s.attributes.get("arguments", {}), sort_keys=True, default=str))
            seen[k] = seen.get(k, 0) + 1
            if seen[k] == 2 and second_idx is None: second_idx = s.index
        if max(seen.values(), default=0) >= 3:
            flagged.append(r.ticket_id)
            wasted += sum(cost_usd(s.name, s.input_tokens, s.output_tokens) for s in r.steps if s.kind == "llm" and second_idx is not None and s.index > second_idx)
    real = {r.ticket_id for r in results if "tool_loop" in r.injected_faults}
    tp = len(real & set(flagged))
    return {"flagged": flagged, "precision": tp / len(flagged) if flagged else 1.0, "recall": tp / len(real) if real else 1.0, "wasted_usd": wasted}

def drift_alert(series, baseline_n=30, ratio=1.6, persist=3):
    import statistics
    base = [x for x in series[:baseline_n] if x == x]
    if len(base) < max(3, baseline_n // 2): return None
    thr, racha = ratio * statistics.median(base), 0
    for i in range(baseline_n, len(series)):
        v = series[i]
        racha = racha + 1 if (v == v and v >= thr) else 0
        if racha >= persist: return i
    return None

def build_finops_dashboard():
    d = Dashboard("finops", "13 · FinOps (proyecto)", tags=["curso", "finops"], description="Coste por ticket, por modelo, caché, desperdicio, presupuesto y desvío.")
    cur = "sum(rate(supportops_ticket_cost_USD_sum[10m])) / sum(rate(supportops_ticket_cost_USD_count[10m]))"
    off = "sum(rate(supportops_ticket_cost_USD_sum[10m] offset 1h)) / sum(rate(supportops_ticket_cost_USD_count[10m] offset 1h))"
    d.add(
        timeseries("Coste por ticket · p95", [prom("histogram_quantile(0.95, sum by (le) (rate(supportops_ticket_cost_USD_bucket[$__rate_interval])))", "p95")], unit="currencyUSD", w=12, description="p95 del coste por ticket (precios de la tabla configurada)."),
        timeseries("Coste por modelo", [prom("sum by (gen_ai_request_model) (rate(supportops_llm_cost_USD_total[$__rate_interval]))", "{{gen_ai_request_model}}")], unit="currencyUSD", stack=True, w=12, description="Dinero por segundo y modelo: un cambio de modelo se ve aquí primero."),
        stat("Hit ratio de la caché", 'sum(rate(supportops_llm_tokens_total{gen_ai_token_type="cache_read"}[10m])) / sum(rate(supportops_llm_tokens_total{gen_ai_token_type=~"input|cache_read|cache_write"}[10m]))', unit="percentunit", w=6, description="Fracción de la entrada servida desde caché. Una caída brusca = caché rota."),
        stat("Gasto desperdiciado", "sum(rate(supportops_ticket_cost_wasted_USD_total[10m])) / sum(rate(supportops_llm_cost_USD_total[10m]))", unit="percentunit", w=6, description="Fracción del gasto en tickets no resueltos, alucinados o en bucle."),
        stat("Burn rate del presupuesto", "max(supportops_budget_burn_rate)", w=6, thr=thresholds((None, "green"), (1, "orange"), (2, "red")), description="1 = justo en presupuesto; 2 = agota el presupuesto a mitad de periodo."),
        timeseries("Desvío del coste por ticket (actual / hace 1 h)", [prom(f"({cur}) / ({off})", "ratio")], w=6, thr=thresholds((None, "green"), (1.6, "red")), description="Cruzar 1,6 durante 5 min dispara la alerta de desvío."),
        logs("Top clientes por coste (Loki)", 'topk(10, sum by (supportops_customer_id) (sum_over_time({service_name="supportops-agent"} |= "ticket costed" | unwrap supportops_finops_cost_usd [$__range])))', w=12, description="Atribución por cliente desde los logs (alta cardinalidad fuera de Prometheus)."),
        traces_table("Tickets caros (Tempo)", '{ name =~ "invoke_agent.*" && span.supportops.finops.cost_usd > 0.002 }', w=12, description="Trazas de tickets con coste superior a 0,002 USD: abre la cascada para ver qué llamadas lo causaron."),
    )
    return d.to_dict()

display(run_rubric())